# Retrieval from ChromaDB

Queries the ChromaDB collection built by `notebooks/ingestion_preprocessing_embedding.ipynb`.

**Run the ingestion notebook first** so `notebooks/chroma_db/` exists on disk.

In [1]:
!pip install chromadb transformers torch


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import chromadb
from transformers import AutoTokenizer, AutoModel
import torch

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_collection("actas_obra")
print(f"Collection loaded with {collection.count()} chunks.")

c:\REPOS\ML problems\POSTGRAU\RAG-UPCSchool-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection loaded with 518 chunks.


In [3]:
model_name = "BSC-LT/MrBERT-es"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 14444.53it/s]
[transformers] ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def embed_query(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().tolist()

In [5]:
search_query = "instalaciones de sistemas y megafonía"

query_embedding = embed_query(search_query)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
)):
    print(f"--- Rank {i+1} | Chunk ID: {meta['chunk_id']} | Distance: {dist:.4f} ---")
    print(doc)
    print()

--- Rank 1 | Chunk ID: 495 | Distance: 937.8229 ---
Edificación Toda la duración de la obra Artículo 11 LOE - El Constructor: El constructor debe ejecutar la obra conforme al proyecto, la normativa aplicable y las instrucciones del director de obra y del director de ejecución de la obra, a fin de alcanzar la calidad exigida en el proyecto. Artículo 12 y 13 LOE - Dirección facultativa: Las modificaciones deben ser informadas por parte de la constructora, y aprobadas por la Dirección Facultativa, previa ejecución de estas. 2.6.

--- Rank 2 | Chunk ID: 81 | Distance: 1051.4974 ---
stión. En visita de 05-11-2025 la UTE confirma que ha realizado el pago del informe, y que en breve enviará el mismo a la DF. En visita de obra de 27-11-2025, la UTE informa que "Drenatges Besós" ha solicitado unos ensayos, como requisito para emitir el informe favorable, y que han realizado estos ensayos. En visita de obra de 10-12-2025, la UTE informa que ha recibido los resultados de los ensayos de compactaci